In [2]:
pip install transformers datasets accelerate torch pandas scikit-learn

In [4]:
import pandas as pd

df = pd.read_csv("empathetic_dialogues.csv")
print(df.head())

   Unnamed: 0                                          Situation      emotion  \
0           0  I remember going to the fireworks with my best...  sentimental   
1           1  I remember going to the fireworks with my best...  sentimental   
2           2  I remember going to the fireworks with my best...  sentimental   
3           3  I remember going to the fireworks with my best...  sentimental   
4           4  I remember going to the fireworks with my best...  sentimental   

                                empathetic_dialogues  \
0  Customer :I remember going to see the firework...   
1  Customer :This was a best friend. I miss her.\...   
2              Customer :We no longer talk.\nAgent :   
3  Customer :Was this a friend you were in love w...   
4             Customer :Where has she gone?\nAgent :   

                                              labels Unnamed: 5 Unnamed: 6  
0  Was this a friend you were in love with, or ju...        NaN        NaN  
1                     

In [5]:
df.columns

Index(['Unnamed: 0', 'Situation', 'emotion', 'empathetic_dialogues', 'labels',
       'Unnamed: 5', 'Unnamed: 6'],
      dtype='object')

In [6]:
df["text"] = (
    "User: " + df["Situation"].astype(str) +
    "\nEmotion: " + df["emotion"].astype(str) +
    "\nAssistant: " + df["empathetic_dialogues"].astype(str)
)

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [8]:
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [9]:
from google.colab import output
output.disable_custom_widget_manager()

In [10]:
  from datasets import Dataset

dataset = Dataset.from_pandas(df[["text"]])

In [11]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/7920 [00:00<?, ? examples/s]

In [12]:
def add_labels(example):
    example["labels"] = example["input_ids"].copy()
    return example

tokenized_dataset = tokenized_dataset.map(add_labels)

Map:   0%|          | 0/7920 [00:00<?, ? examples/s]

In [13]:
import torch
from transformers import TrainingArguments

# The Trainer automatically moves the model to GPU if it's available.
# Enabling fp16=True is the best optimization for a T4 GPU.
training_args = TrainingArguments(
    output_dir="./mental_health_model",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    save_steps=5000,
    logging_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(), # Use mixed precision on T4 for speed
    report_to="none"
)

print(f"🚀 Training on: {'GPU (T4)' if torch.cuda.is_available() else 'CPU'}")
print(f"⚡ Mixed Precision (fp16): {training_args.fp16}")

🚀 Training on: GPU (T4)
⚡ Mixed Precision (fp16): True


In [14]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

trainer.save_model("mental_health_chatbot")
tokenizer.save_pretrained("mental_health_chatbot")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,2.763126
200,2.488091
300,2.504686
400,2.427994
500,2.410091
600,2.396584
700,2.304735
800,2.334350
900,2.287087
1000,2.250772


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('mental_health_chatbot/tokenizer_config.json',
 'mental_health_chatbot/tokenizer.json')

In [15]:
# Run this cell in your Google Colab notebook to find the right prompt format!

test_input = "I am tired today"

# The 4 most common formatting patterns used in datasets
variants = {
    "Variant A (Space-Colon)": f"Customer : {test_input}\nAgent : ",
    "Variant B (No-Space-Colon)": f"Customer: {test_input}\nAgent:",
    "Variant C (Continuous String)": f"Customer: {test_input} Agent:",
    "Variant D (Raw Context Shift)": f"{test_input} \n "
}

print("--- TESTING PROMPT FORMATS ---")
for name, prompt in variants.items():
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False, # Using False (greedy search) to see its absolute strongest bias
            pad_token_id=tokenizer.eos_token_id
        )

    raw_response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n{name} Output:")
    print(f"-> '{raw_response.strip()}'")

--- TESTING PROMPT FORMATS ---

Variant A (Space-Colon) Output:
-> 'I am tired today
Agent :I am tired today
Agent :I am tired today
Agent :I am tired today
Agent :I am'

Variant B (No-Space-Colon) Output:
-> 'Customer :I am tired today
Agent :I am tired today
Agent :I am tired today
Agent :I am tired today
Agent :'

Variant C (Continuous String) Output:
-> 'Customer :I am tired today Agent: I am tired today Agent: I am tired today Agent:I am tired today Agent:I am tired today'

Variant D (Raw Context Shift) Output:
-> 'Emotion: tired
Assistant: Customer :I am tired today 
Agent :I am tired today 
Agent :I am tired today'


In [16]:
test_input = "I am tired today"

# The exact character-for-character prefix matching your training fingerprint
perfect_prompt = f"Emotion: tired\nAssistant: Customer :{test_input}\nAgent :"

inputs = tokenizer(perfect_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.5,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip())

That's a long time.  I hope you have a good day next week! :) Hope you got it all the right way! (Laughs)
Agent ::Yea, that's great to


In [17]:
!pip install streamlit -q
!npm install -q -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 47.8 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧

In [25]:
%%writefile app.py
import streamlit as st
import torch
import re
from transformers import AutoModelForCausalLM, AutoTokenizer

st.set_page_config(page_title="Mental Health AI Assistant", page_icon="🤖")
st.title("Mental Health AI Assistant")
st.write("Running live on Colab's GPU cloud!")

MODEL_PATH = "mental_health_chatbot"

@st.cache_resource
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH).to(device)
    return tokenizer, model, device

tokenizer, model, device = load_model()

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if user_input := st.chat_input("How can I help you today?"):
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)

    with st.chat_message("assistant"):
        response_placeholder = st.empty()
        response_placeholder.markdown("Thinking...")

        formatted_prompt = f"Emotion: sad\nAssistant: Customer :{user_input}\nAgent :"

        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

        # Perfect 8-space indentation here
        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                max_new_tokens=35,          # Tighter response length
                do_sample=True,
                temperature=0.4,            # More focused
                top_p=0.9,
                repetition_penalty=1.3,     # Aggressive loop prevention
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id
            )

        input_length = inputs.input_ids.shape[1]
        raw_response = tokenizer.decode(output_tokens[0][input_length:], skip_special_tokens=True).strip()

        clean_response = re.split(r"(Customer|Agent|Emotion|Assistant)\s*[:\(\?\s,]", raw_response, flags=re.IGNORECASE)[0].strip()
        for word in ["Agent", "Customer", "Emotion", "Assistant"]:
            clean_response = clean_response.replace(word, "")
        clean_response = clean_response.strip()

        if not clean_response or len(clean_response) < 3:
            clean_response = "I hear you. Tell me more about how you're feeling right now."

        response_placeholder.markdown(clean_response)
        st.session_state.messages.append({"role": "assistant", "content": clean_response})

Overwriting app.py


In [20]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-06-02 09:34:12--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.5.2/cloudflared-linux-amd64.deb [following]
--2026-06-02 09:34:12--  https://github.com/cloudflare/cloudflared/releases/download/2026.5.2/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/0ec93a74-a3bc-474a-bd4b-e852ababcbb3?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-02T10%3A08%3A08Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

In [ ]:

!streamlit run app.py --server.port 8501 --server.enableCORS=false --server.enableXsrfProtection=false & cloudflared tunnel --url http://localhost:8501

2026-06-02T09:40:28Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-02T09:40:28Z INF Requesting new quick Tunnel on trycloudflare.com...


2026-06-02 09:40:30.286 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.16.201.157:8501

2026-06-02T09:40:30Z INF +--------------------------------------------------------